# 05 - Bronze profiling

The dbt models (`swiss-mobility-dbt/models`) were designed before any data was collected. This notebook checks their assumptions against the bronze data written by the Azure Function:

| # | Assumption | Used by |
|---|---|---|
| 1 | `station_id + category + line_number + departure_planned_utc` identifies one departure | dedup in `stg_transport__departures`, `departure_id` |
| 2 | `delay_min` is null only when no real-time data exists | punctuality measures |
| 3 | every `category` is listed in the seed `transport_categories.csv` | `dim_line`, transport mode |
| 4 | cancelled departures can be identified | punctuality (a cancelled train is not "on time") |

The bronze files are copied from ADLS into `data/adls_snapshot/` (git-ignored). Only files not yet present are downloaded, so the notebook can be re-run as data accumulates.

In [1]:
import json
from pathlib import Path

import pandas as pd
import pyarrow.dataset as ds

from ingestion.config import ADLS_ACCOUNT_NAME, BRONZE_CONTAINER, DEPARTURES_DATASET, INGESTION_LOG_DATASET
from ingestion.transform import DEPARTURES_SCHEMA, INGESTION_LOG_SCHEMA

SNAPSHOT = Path("/workspace/data/adls_snapshot")
SEED = Path("/workspace/swiss-mobility-dbt/seeds/transport_categories.csv")
DEDUP_KEY = ["station_id", "category", "line_number", "departure_planned_utc"]

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)

## Snapshot of the bronze container

Authentication uses `DefaultAzureCredential`, i.e. the service principal configured in `.env` (`AZURE_CLIENT_ID`, `AZURE_TENANT_ID`, `AZURE_CLIENT_SECRET`), which has read access to the storage account.

In [2]:
def download_new_files(dataset):
    """Copy the dataset's Parquet files from ADLS to SNAPSHOT, skipping files already present."""
    from azure.identity import DefaultAzureCredential
    from azure.storage.filedatalake import DataLakeServiceClient

    service = DataLakeServiceClient(
        account_url=f"https://{ADLS_ACCOUNT_NAME}.dfs.core.windows.net",
        credential=DefaultAzureCredential(),
    )
    filesystem = service.get_file_system_client(BRONZE_CONTAINER)
    downloaded = present = 0
    for item in filesystem.get_paths(path=dataset, recursive=True):
        if item.is_directory or not item.name.endswith(".parquet"):
            continue
        target = SNAPSHOT / item.name
        if target.exists():
            present += 1
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(filesystem.get_file_client(item.name).download_file().readall())
        downloaded += 1
    print(f"{dataset}: {downloaded} downloaded, {present} already present")


for dataset in (DEPARTURES_DATASET, INGESTION_LOG_DATASET):
    download_new_files(dataset)

transport_stationboard: 518 downloaded, 0 already present
_ingestion_log: 518 downloaded, 0 already present


## Load

The datasets are read with their explicit schema: files written before a column was added (e.g. `station_list_as_of`) get nulls for it instead of the column being dropped.

In [3]:
def load(dataset, schema):
    table = ds.dataset(SNAPSHOT / dataset, format="parquet", schema=schema, partitioning="hive").to_table()
    return table.to_pandas()


departures = load(DEPARTURES_DATASET, DEPARTURES_SCHEMA)
runs = load(INGESTION_LOG_DATASET, INGESTION_LOG_SCHEMA)

overview = pd.Series({
    "departure rows": len(departures),
    "batches": departures["_batch_id"].nunique(),
    "stations": departures["station_id"].nunique(),
    "first batch": departures["_batch_id"].min(),
    "last batch": departures["_batch_id"].max(),
    "planned departures from": departures["departure_planned_utc"].min(),
    "planned departures to": departures["departure_planned_utc"].max(),
})
overview

departure rows                                827616
batches                                          518
stations                                          16
first batch                         20261023T133136Z
last batch                          20261029T200000Z
planned departures from    2026-10-23 13:31:00+00:00
planned departures to      2026-10-30 17:11:00+00:00
dtype: object

### Ingestion runs

Share of successful station calls and origin of the station list, per day.

In [4]:
runs["day"] = runs["started_at"].dt.date
runs.groupby("day").agg(
    batches=("_batch_id", "nunique"),
    station_calls=("station_id", "size"),
    failed=("status", lambda s: (s == "failed").sum()),
    from_cache=("station_source", lambda s: (s == "cache").sum()),
)

,batches,station_calls,failed,from_cache
day,,,,
2026-10-23,1,16,0,0
2026-10-24,53,848,12,0
2026-10-25,96,1536,12,0
2026-10-26,96,1536,6,0
2026-10-27,96,1536,0,0
2026-10-28,96,1536,0,0
2026-10-29,80,1280,0,0


In [5]:
# Failed station calls, if any
runs.loc[runs["status"] == "failed", ["_batch_id", "station_name", "attempts", "error"]]

,_batch_id,station_name,attempts,error
816,20261024T231500Z,Basel SBB,3,ConnectTimeout after 3 attempts
817,20261024T231500Z,Delémont,3,ConnectTimeout after 3 attempts
818,20261024T231500Z,Olten,3,ConnectTimeout after 3 attempts
819,20261024T231500Z,Genève,3,ConnectTimeout after 3 attempts
820,20261024T231500Z,Lausanne,3,ConnectTimeout after 3 attempts
821,20261024T231500Z,Sion,3,ConnectTimeout after 3 attempts
822,20261024T231500Z,Zürich HB,3,ConnectTimeout after 3 attempts
823,20261024T231500Z,Fribourg/Freiburg,3,ConnectTimeout after 3 attempts
824,20261024T231500Z,Neuchâtel,3,ConnectionError after 3 attempts
825,20261024T231500Z,Biel/Bienne,3,ConnectionError after 3 attempts


## 1. Dedup key

**Check A: within one batch.** A poll lists each departure once, so two rows with the same key in the same batch are two *different* departures sharing the key. The dedup in staging would drop one of them.

In [6]:
within_batch = departures.groupby(["_batch_id", *DEDUP_KEY], dropna=False).size()
collisions = within_batch[within_batch > 1]

print(f"Keys appearing more than once within a batch: {len(collisions)} "
      f"({collisions.sum()} rows out of {len(departures)})")

Keys appearing more than once within a batch: 4519 (9038 rows out of 827616)


In [7]:
# Examples: every column of the colliding rows, to see what distinguishes them
colliding_keys = collisions.reset_index()[["_batch_id", *DEDUP_KEY]].head(10)
departures.merge(colliding_keys, on=["_batch_id", *DEDUP_KEY], how="inner", validate="many_to_many") \
    .sort_values(["_batch_id", *DEDUP_KEY]).head(30)

,station_id,category,line_number,operator,destination,departure_planned_utc,departure_planned_raw,delay_min,platform,_ingested_at,_batch_id
0,8506302,S,21,AB-ab,Trogen,2026-10-23 13:56:00+00:00,2026-10-23T15:56:00+0200,0.0,12,2026-10-23 13:31:36+00:00,20261023T133136Z
1,8506302,S,21,AB-ab,Appenzell,2026-10-23 13:56:00+00:00,2026-10-23T15:56:00+0200,0.0,11,2026-10-23 13:31:36+00:00,20261023T133136Z
4,8506302,S,21,AB-ab,Appenzell,2026-10-23 14:26:00+00:00,2026-10-23T16:26:00+0200,0.0,11,2026-10-23 13:31:36+00:00,20261023T133136Z
5,8506302,S,21,AB-ab,Trogen,2026-10-23 14:26:00+00:00,2026-10-23T16:26:00+0200,0.0,12,2026-10-23 13:31:36+00:00,20261023T133136Z
8,8506302,S,21,AB-ab,Trogen,2026-10-23 14:56:00+00:00,2026-10-23T16:56:00+0200,0.0,12,2026-10-23 13:31:36+00:00,20261023T133136Z
9,8506302,S,21,AB-ab,Appenzell,2026-10-23 14:56:00+00:00,2026-10-23T16:56:00+0200,NaN,11,2026-10-23 13:31:36+00:00,20261023T133136Z
12,8506302,S,21,AB-ab,Appenzell,2026-10-23 15:26:00+00:00,2026-10-23T17:26:00+0200,0.0,11,2026-10-23 13:31:36+00:00,20261023T133136Z
13,8506302,S,21,AB-ab,Trogen,2026-10-23 15:26:00+00:00,2026-10-23T17:26:00+0200,0.0,12,2026-10-23 13:31:36+00:00,20261023T133136Z
16,8506302,S,21,AB-ab,Appenzell,2026-10-23 15:56:00+00:00,2026-10-23T17:56:00+0200,0.0,11,2026-10-23 13:31:36+00:00,20261023T133136Z
17,8506302,S,21,AB-ab,Trogen,2026-10-23 15:56:00+00:00,2026-10-23T17:56:00+0200,0.0,12,2026-10-23 13:31:36+00:00,20261023T133136Z


In [8]:
# Which categories are affected, and is line_number empty there?
if len(collisions):
    affected = collisions.reset_index()
    print(affected.groupby("category", dropna=False).size().sort_values(ascending=False))
    print("\nNull line_number among colliding keys:", affected["line_number"].isna().sum())

category
S     2831
IC    1123
R      204
IR     169
RE     154
SN      38
dtype: int64

Null line_number among colliding keys: 0


**Candidate key with the destination.** The colliding rows above differ by destination (and platform). The destination is part of the timetable; the platform could change between polls. The cell below measures both: collisions left with the destination in the key, and how often the platform of a departure changes between polls.

In [ ]:
NEW_KEY = [*DEDUP_KEY, "destination"]

within_batch_new = departures.groupby(["_batch_id", *NEW_KEY], dropna=False).size()
print(f"Collisions within a batch with destination in the key: {(within_batch_new > 1).sum()}")

platforms_per_train = departures.groupby(NEW_KEY, dropna=False)["platform"].nunique()
print(f"Trains whose platform changed between polls: {(platforms_per_train > 1).sum()} "
      f"of {len(platforms_per_train)}")

**Remaining collisions.** Same departure listed twice in one poll, or two departures still sharing the key? Count, per group, the columns whose values differ between the colliding rows.

In [ ]:
remaining = within_batch_new[within_batch_new > 1].reset_index()[["_batch_id", *NEW_KEY]]
rows = departures.merge(remaining, on=["_batch_id", *NEW_KEY])

other_columns = ["operator", "platform", "delay_min", "departure_planned_raw"]
differs = rows.groupby(["_batch_id", *NEW_KEY], dropna=False)[other_columns].nunique(dropna=False)
print("Groups where each column differs between the colliding rows:")
print((differs > 1).sum())

rows.sort_values(["_batch_id", *NEW_KEY]).head(12)

**Check B: across batches.** The same physical departure (same station, scheduled minute and destination) should keep the same key from one poll to the next. Several keys for one departure would be counted as several departures after deduplication.

In [9]:
physical = ["station_id", "departure_planned_utc", "destination"]
keys_per_departure = (
    departures.drop_duplicates([*physical, "category", "line_number"])
    .groupby(physical, dropna=False)
    .size()
)
unstable = keys_per_departure[keys_per_departure > 1]
print(f"Departures seen under more than one key: {len(unstable)} of {len(keys_per_departure)}")

departures.merge(unstable.reset_index()[physical].head(10), on=physical) \
    .sort_values([*physical, "_ingested_at"])[[*physical, "category", "line_number", "operator", "_batch_id"]].head(30)

Departures seen under more than one key: 30 of 44256


,station_id,departure_planned_utc,destination,category,line_number,operator,_batch_id
0,8500010,2026-10-24 17:07:00+00:00,Zürich HB,IR,36,SBB,20261024T131500Z
1,8500010,2026-10-24 17:07:00+00:00,Zürich HB,ICE,000109,SBB,20261024T131500Z
2,8500010,2026-10-24 17:07:00+00:00,Zürich HB,IR,36,SBB,20261024T133000Z
3,8500010,2026-10-24 17:07:00+00:00,Zürich HB,ICE,000109,SBB,20261024T133000Z
4,8500010,2026-10-24 17:07:00+00:00,Zürich HB,ICE,000109,SBB,20261024T134500Z
5,8500010,2026-10-24 17:07:00+00:00,Zürich HB,IR,36,SBB,20261024T134500Z
6,8500010,2026-10-24 17:07:00+00:00,Zürich HB,IR,36,SBB,20261024T140000Z
7,8500010,2026-10-24 17:07:00+00:00,Zürich HB,ICE,000109,SBB,20261024T140000Z
8,8500010,2026-10-24 17:07:00+00:00,Zürich HB,IR,36,SBB,20261024T141500Z
9,8500010,2026-10-24 17:07:00+00:00,Zürich HB,ICE,000109,SBB,20261024T141500Z


In [10]:
# How many times is a departure polled before it leaves? (expected: several, every 15 minutes)
polls = departures.groupby(DEDUP_KEY, dropna=False)["_batch_id"].nunique()
polls.describe()

count    43934.000000
mean        18.734852
std         16.867418
min          1.000000
25%          6.000000
50%         14.000000
75%         25.000000
max         88.000000
Name: _batch_id, dtype: float64

## 2. Missing delays

`delay_min` is null when the API has no real-time information. If nulls are concentrated in some categories (e.g. buses), punctuality for those categories cannot be measured and must not be reported as 0 minutes.

In [11]:
latest = departures.sort_values("_ingested_at").drop_duplicates(DEDUP_KEY, keep="last")

delay_by_category = latest.groupby("category", dropna=False).agg(
    departures=("delay_min", "size"),
    delay_known=("delay_min", lambda s: s.notna().sum()),
)
delay_by_category["known_share"] = (delay_by_category["delay_known"] / delay_by_category["departures"]).round(3)
delay_by_category.sort_values("departures", ascending=False)

,departures,delay_known,known_share
category,,,
S,19202,17947,0.935
IR,8326,7470,0.897
IC,5476,4739,0.865
R,5103,4321,0.847
RE,4089,3696,0.904
EC,626,532,0.850
ICE,380,172,0.453
TER,325,292,0.898
TGV,165,155,0.939


In [12]:
# Distribution of known delays (minutes)
latest["delay_min"].describe(percentiles=[0.5, 0.9, 0.95, 0.99])

count    39497.000000
mean         0.272147
std          1.199952
min          0.000000
50%          0.000000
90%          1.000000
95%          1.000000
99%          3.000000
max         58.000000
Name: delay_min, dtype: float64

In [13]:
# Negative or extreme values, checked by tests/assert_delay_within_bounds.sql in dbt
latest.loc[(latest["delay_min"] < 0) | (latest["delay_min"] > 180),
           ["station_id", "category", "line_number", "departure_planned_utc", "delay_min"]]

,station_id,category,line_number,departure_planned_utc,delay_min


## 3. Categories versus the seed

Categories missing from `transport_categories.csv` would have no transport mode in `dim_line`.

In [14]:
seed = pd.read_csv(SEED)
observed = latest["category"].str.strip().str.upper().value_counts(dropna=False).rename("departures")

comparison = observed.to_frame().join(seed.set_index("category"), how="outer")
comparison["status"] = "ok"
comparison.loc[comparison["transport_mode"].isna(), "status"] = "missing from seed"
comparison.loc[comparison["departures"].isna(), "status"] = "in seed, not observed"
comparison.sort_values(["status", "departures"], ascending=[True, False])

,departures,transport_mode,category_label,status
category,,,,
BAT,NaN,boat,Boat,"in seed, not observed"
BN,NaN,bus,Night bus,"in seed, not observed"
FUN,NaN,funicular,Funicular,"in seed, not observed"
T,NaN,tram,Tram,"in seed, not observed"
TER,325.0,NaN,NaN,missing from seed
TGV,165.0,NaN,NaN,missing from seed
PE,109.0,NaN,NaN,missing from seed
NJ,35.0,NaN,NaN,missing from seed
RJX,10.0,NaN,NaN,missing from seed


## 4. Cancellations

The bronze schema has no cancellation column. The cell below calls the API once for a large station and lists the fields returned for each departure, to see whether cancellations are exposed (e.g. in `stop.prognosis` or a dedicated flag) and whether the ingestion should capture them.

In [15]:
import requests

response = requests.get(
    "https://transport.opendata.ch/v1/stationboard",
    params={"id": "8503000", "limit": 100},
    timeout=15,
)
board = response.json()["stationboard"]

def field_paths(obj, prefix=""):
    """All key paths of a nested dict, e.g. stop.prognosis.departure."""
    paths = set()
    if isinstance(obj, dict):
        for key, value in obj.items():
            path = f"{prefix}{key}"
            paths.add(path)
            paths |= field_paths(value, path + ".")
    return paths

all_paths = sorted(set().union(*(field_paths(dep) for dep in board)))
[p for p in all_paths if not p.startswith(("passList", "stop.station.coordinate", "stop.location"))]

['capacity1st',
 'capacity2nd',
 'category',
 'categoryCode',
 'name',
 'number',
 'operator',
 'stop',
 'stop.arrival',
 'stop.arrivalTimestamp',
 'stop.delay',
 'stop.departure',
 'stop.departureTimestamp',
 'stop.platform',
 'stop.prognosis',
 'stop.prognosis.arrival',
 'stop.prognosis.capacity1st',
 'stop.prognosis.capacity2nd',
 'stop.prognosis.departure',
 'stop.prognosis.platform',
 'stop.realtimeAvailability',
 'stop.station',
 'stop.station.distance',
 'stop.station.id',
 'stop.station.name',
 'stop.station.score',
 'subcategory',
 'to']

In [16]:
# Values of the candidate fields across the 100 departures
candidates = [p for p in all_paths if any(word in p.lower() for word in ("cancel", "prognosis", "capacity"))]

def get_path(obj, path):
    for part in path.split("."):
        obj = obj.get(part) if isinstance(obj, dict) else None
    return obj

pd.DataFrame({p: [json.dumps(get_path(dep, p)) for dep in board] for p in candidates}).apply(pd.Series.value_counts).head(15)

,capacity1st,capacity2nd,stop.prognosis,stop.prognosis.arrival,stop.prognosis.capacity1st,stop.prognosis.capacity2nd,stop.prognosis.departure,stop.prognosis.platform
"""2026-10-29T21:12:36+0100""",NaN,NaN,NaN,91.0,NaN,NaN,NaN,NaN
"""2026-10-29T21:13:00+0100""",NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN
"""2026-10-29T21:13:36+0100""",NaN,NaN,NaN,8.0,NaN,NaN,NaN,NaN
"""2026-10-29T21:14:00+0100""",NaN,NaN,NaN,NaN,NaN,NaN,3.0,NaN
"""2026-10-29T21:14:36+0100""",NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN
"""2026-10-29T21:15:00+0100""",NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN
"""2026-10-29T21:16:00+0100""",NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN
"""2026-10-29T21:17:00+0100""",NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN
"""2026-10-29T21:18:00+0100""",NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN
"""2026-10-29T21:19:00+0100""",NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN


In [ ]:
# realtimeAvailability: does it explain missing real-time data (e.g. a cancellation)?
pd.Series([get_path(dep, "stop.realtimeAvailability") for dep in board]).value_counts(dropna=False)

## Observations

Data profiled: 518 batches (23-29 October 2026, 827,616 rows, 16 stations). The key checks were repeated on 11 days of data with the same conclusions.

**Ingestion runs**
- 96 batches on every full day: no scheduled run was missed. The station list always came from Oracle (cache never used).
- Failures are grouped in one batch per night (22:30-00:15 local time): all stations fail with connection timeouts for a few minutes, then the API answers again. On 24 and 25 October, 12 of 16 stations failed, the run failed and the failed-run alert was received. The outage is on the API side.

**1. Dedup key**
- With `station + category + line + scheduled time`, 4,519 keys appeared twice within one batch (S, IC, R, IR, RE, SN). Example: S21 at St. Gallen leaves at the same minute to Trogen (platform 12) and to Appenzell (platform 11). The key merged two different departures.
- Adding the destination leaves 72 collisions. All of them are one departure listed twice in the same poll: 38 identical copies, 34 split copies (one with the delay and no platform, one with the platform and no delay). The platform of a departure changed between polls for only 2 of 81,535 departures (`stop.platform` is the scheduled platform; live changes are in `stop.prognosis.platform`).
- Check B: 30 of 44,256 departures appear under two names in every poll (e.g. IR 36 and ICE 109, Basel SBB to Zürich HB at the same minute), most likely one cross-border train with two train numbers. Keys are stable between polls.
- A departure is polled 14 times on median before it leaves.

**2. Missing delays**
- Share of departures with a known delay: S 93.5 %, RE 90.4 %, IR 89.7 %, IC 86.5 %, EC 85.0 %, R 84.7 %; ICE 45.3 %, SN 32.1 %, bus 0 % (6 departures).
- Known delays: median 0 min, 95th percentile 1 min, 99th percentile 3 min, maximum 58 min. No negative values.

**3. Categories**
- Missing from the seed: TER (325), TGV (165), PE (109), NJ (35), RJX (10), EXT (1), all trains, mostly international.
- In the seed but not observed: BAT, BN, FUN, T. The 16 stations are railway stations.

**4. Cancellations**
- The stationboard has no cancellation field, and `stop.realtimeAvailability` is empty for every departure. Cancellations cannot be identified from this source.

**Decisions applied to the dbt project**
- Dedup key extended with the destination (`stg_transport__departures`, `departure_id`); ties within a poll keep the copy with a known delay, then with a platform.
- Six train categories added to `transport_categories.csv`.
- `assert_delay_within_bounds` fails on delays below 0 or above 300 minutes.
- Accepted and documented: departures with two train numbers (0.07 %), no cancellation data. Punctuality is reported with its real-time coverage, since coverage is low for ICE and SN.
